# Script 2

#### Objective
The next notebook takes info from CVs, transforms this info to its vector embedding representation, feeds this data into a database, and allows searching in the database for the CVs that best suit a query across multiple vector embedding models.

### 1. Initializing a local client

Qdrant db client can work in the cloud or locally in hr_three ways: In-memory, on hard drive, and in a docker container. Here ":memory:" mode will be used for testing purposes so the database lives in RAM.

In [1]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

### 2. Creating collections

A Qdrant collection is an isolated container holding vectors, IDs, and metadata payloads. We create a separate collection for each model to evaluate and compare them independently.

In [ ]:
models_to_test = {
    "GIST-all-MiniLM-L6-v2": {"size": 384, "model_name": "avsolatorio/GIST-all-MiniLM-L6-v2"},
    "bge-base-en-v1.5": {"size": 768, "model_name": "BAAI/bge-base-en-v1.5"},
    "harrier-oss-v1-0.6b": {"size": 1024, "model_model_name": "microsoft/harrier-oss-v1-0.6b"},
    "Qwen3-Embedding-4B": {"size": 2560, "model_name": "Qwen/Qwen3-Embedding-4B"}
}

# Create a unique collection for each model
for name, config in models_to_test.items():
    COLLECTION_NAME = f"test_{name}"
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=config["size"],
            distance=models.Distance.COSINE
        )
    )
print("Collections created successfully!")

Collections created successfully!


### 3. Loading CV data and creating uniform data_points

To fairly compare all vector embedding models, we generate the exact same `data_points` (chunked CV snippets) using a reference model before encoding them with each model.

In [3]:
import json

with open("../data/cv_extracted_info_eng.json", "r", encoding="utf-8") as file:
    cv_data = json.load(file)

print(f"Loaded {len(cv_data)} CVs")

Loaded 91 CVs


Loading reference model to calculate token length constraints for uniform chunking across models.

In [4]:
import os
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

# Load environment variables from the .env file in the root directory
load_dotenv("../.env")
HUGGING_FACE_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

# Load reference model to build fixed data_points
ref_model_name = models_to_test["GIST-all-MiniLM-L6-v2"]["model_name"]
ref_model = SentenceTransformer(ref_model_name, token=HUGGING_FACE_API_KEY, trust_remote_code=True)

/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10341.43it/s]


Constructing uniform `data_points` for all embedding models.

In [5]:
def is_less_than_max_tokens(model, text, applicant_name, chunk_num):
    num_tokens = len(model.tokenizer.encode(text))
    max_tokens = model.max_seq_length
    if num_tokens > max_tokens:
        return False
    return True

max_tokens = ref_model.max_seq_length

data_points = []
for idx, cv in enumerate(cv_data):
    # Fields like "name", "profession", "email", "linkedin" and "phone" can go in the metadata
    
    # First CV chunk
    chunk = (
        f"Candidate name: {cv['name']} | "
        f"Candidate profession: {cv['profession']} | "
        f"Content: (about_me: {cv['about_me']}, seniority_level: {cv['seniority_level']}, location: {cv['location']}, experience_years: {cv['experience_years']}, languages: {cv.get('languages', [])})"
    )
    if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 1)):
        data_points.append({
            "chunk": chunk,
            "chunk_meta": "about_me-seniority_level-location-experience_years-languages", # unique identifier of the chunk
            "cv": cv
        })
    else:
        pass

    # Second CV chunk
    chunk = (
        f"Candidate name: {cv['name']} | "
        f"Candidate profession: {cv['profession']} | "
        f"Content: (certifications: {cv['certifications']}, education: {cv['education']}, skills: {cv['skills']})"
    )
    if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 2)):
        data_points.append({
            "chunk": chunk,
            "chunk_meta": "certifications-education-skills", # unique identifier of the chunk
            "cv": cv
        })
    else:
        # Separate into three chunks
        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (certifications: {cv['certifications']})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 2.1)):
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "certifications", # unique identifier of the chunk
                "cv": cv
            })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 2.1 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Attaching anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "certifications", # unique identifier of the chunk
                "cv": cv
            })

        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (education: {cv['education']})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 2.2)):
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "education", # unique identifier of the chunk
                "cv": cv
            })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 2.2 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Appending data anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "education", # unique identifier of the chunk
                "cv": cv
            })

        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (skills: {cv['skills']})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 2.3)):
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "skills", # unique identifier of the chunk
                "cv": cv
            })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 2.3 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Appending data anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "skills", # unique identifier of the chunk
                "cv": cv
            })

    # Third chunk
    chunk = (
        f"Candidate name: {cv['name']} | "
        f"Candidate profession: {cv['profession']} | "
        f"Content: (experience: {cv['experience']})"
    )
    if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 3)):
        data_points.append({
            "chunk": chunk,
            "chunk_meta": "experience", # unique identifier of the chunk
            "cv": cv
        })
    else:
        # Split experience in half
        exp = f"experience_1: {cv['experience']}"
        half = int(len(exp)/2)
        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (experience_part_1: {exp[:half]})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 3.1)):
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "experience_part_1", # unique identifier of the chunk
                "cv": cv
            })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 3.1 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Appending data anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "experience_part_1", # unique identifier of the chunk
                "cv": cv
            })

        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (experience_part_2: {exp[half:]})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 3.2)):
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "experience_part_2", # unique identifier of the chunk
                "cv": cv
            })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 3.2 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Appending data anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "experience_part_2", # unique identifier of the chunk
                "cv": cv
            })

print(f"Total data_points created: {len(data_points)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (561 > 512). Running this sequence through the model will result in indexing errors


serialized CV for JUAN SANTIAGO VILLEGAS LÓPEZ, chunk 2.1 has 514 tokens, which is greater than the number of tokens this model supports (512). Attaching anyways...
serialized CV for Deivy Stiven Hernandez Castañeda, chunk 3.1 has 519 tokens, which is greater than the number of tokens this model supports (512). Appending data anyways...
serialized CV for Deivy Stiven Hernandez Castañeda, chunk 3.2 has 525 tokens, which is greater than the number of tokens this model supports (512). Appending data anyways...
Total data_points created: 303


### 4. Vector Embedding the CVs and Uploading to Qdrant Collections

In [8]:
loaded_models = {}

for name, config in models_to_test.items():
    COLLECTION_NAME = f"test_{name}"
    print(f"\n==========================================")
    print(f"Processing model: {name} ({config['model_name']})")
    print(f"==========================================")
    
    model = SentenceTransformer(config["model_name"], token=HUGGING_FACE_API_KEY, trust_remote_code=True)
    loaded_models[name] = model
    
    embeddings = model.encode([point["chunk"] for point in data_points])
    print(f"Embeddings shape for {name}: {embeddings.shape}")
    embeddings_list = embeddings.tolist()
    
    client.upload_points(
        collection_name=COLLECTION_NAME,
        points=[
            models.PointStruct(
                id=idx,
                vector=embeddings_list[idx],
                payload={
                    **data_point['cv'],
                    "chunk": data_point['chunk']
                }
            )
            for idx, data_point in enumerate(data_points)
        ],
    )
    print(f"Successfully inserted {len(data_points)} points into collection '{COLLECTION_NAME}'")


Processing model: GIST-all-MiniLM-L6-v2 (avsolatorio/GIST-all-MiniLM-L6-v2)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11484.22it/s]


Embeddings shape for GIST-all-MiniLM-L6-v2: (303, 384)
Successfully inserted 303 points into collection 'test_GIST-all-MiniLM-L6-v2'

Processing model: bge-base-en-v1.5 (BAAI/bge-base-en-v1.5)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24138.66it/s]


Embeddings shape for bge-base-en-v1.5: (303, 768)
Successfully inserted 303 points into collection 'test_bge-base-en-v1.5'

Processing model: harrier-oss-v1-0.6b (microsoft/harrier-oss-v1-0.6b)


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 22717.47it/s]


Embeddings shape for harrier-oss-v1-0.6b: (303, 1024)
Successfully inserted 303 points into collection 'test_harrier-oss-v1-0.6b'

Processing model: Qwen3-Embedding-4B (Qwen/Qwen3-Embedding-4B)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 8883.49it/s]


Embeddings shape for Qwen3-Embedding-4B: (303, 2560)
Successfully inserted 303 points into collection 'test_Qwen3-Embedding-4B'


### 5. Querying the Database and Comparing Models

In [9]:
with open("../data/job_descriptions_train_batch.json", "r", encoding="utf-8") as f:
    job_descriptions = json.load(f)

job_descriptions

[{'id': 'jd_ai_agents_engineer',
  'title': 'Founding AI Agents Engineer & Automations Developer',
  'description': 'Senior hybrid role in Medellín ($5M–7M COP/month) with 5+ years experience. Responsible for designing, deploying, and maintaining autonomous AI agents and low-code digital workflows using n8n, LangGraph, CrewAI, MCP, RAG, Python, and LLMs (GPT-4, Claude 3.5, Gemini). Connects APIs with legacy CRMs (Bitrix24, Zoho) and establishes productized SaaS formulas. Requires fluent technical English.',
  'source': 'AI Agents Developer & AI Strategist (Founding Team) _ The 100x Company _ LinkedIn.pdf'},
 {'id': 'jd_btl_marketing_coordinator',
  'title': 'BTL Marketing Coordinator',
  'description': 'Mid-level role in Medellín ($7.2M COP/month, 3 yrs exp) designing and executing national BTL strategies and brand events. Responsibilities include approving distributor proposals, overseeing advertising merchandising, coordinating event logistics, managing BTL auxiliary staff, and manag

Loading ground truth matrix and evaluating query results across all embedding model collections. General evaluation by printing the score given in the benchmark for just the first job description.

In [23]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [39]:
with open("../data/ground_truth_matrix.json", "r", encoding="utf-8") as f:
    ground_truth_matrix = json.load(f)

jd_id = "jd_ai_agents_engineer"
jd_query = job_descriptions[0]["description"]
eval_benchmarks = ground_truth_matrix[jd_id]["evaluations"]

print(f"=== Query Evaluation for Job Description: '{job_descriptions[0]['title']}' ({jd_id}) ===\n")
top_k_results = {}
for name, config in models_to_test.items():
    top_k_results[name] = {}
    COLLECTION_NAME = f"test_{name}"
    model = loaded_models[name]
    
    query_vector = model.encode(jd_query).tolist()
    
    result = client.query_points_groups(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=10,
        group_by="email",
        group_size=1
    )
    
    print("\n" + "="*80)
    print(f" MODEL: {name} | COLLECTION: {COLLECTION_NAME}")
    print("="*80)
    
    for group in result.groups:
        top_k_results[name]["group_id"] = group.id
        eval_matches = [e for e in eval_benchmarks if e['email'] == group.id]
        if eval_matches:
            eval_item = eval_matches[0]
            top_k_results[name]["group_id"] = group.id
            benchmark_info = f"Benchmark Score: {eval_item['score']} | reasoning: {eval_item['reasoning']}"
        else:
            benchmark_info = "No benchmark evaluation match found"
            
        print(f"Group ID (Email): {group.id}")
        for hit in group.hits:
            print(f"  - Point ID: {hit.id}, Score: {hit.score:.4f}")
            print(f"  - {benchmark_info}")

=== Query Evaluation for Job Description: 'Founding AI Agents Engineer & Automations Developer' (jd_ai_agents_engineer) ===


 MODEL: GIST-all-MiniLM-L6-v2 | COLLECTION: test_GIST-all-MiniLM-L6-v2
Group ID (Email): lujan9495@gmail.com
  - Point ID: 87, Score: 0.8494
  - Benchmark Score: 1 | reasoning: The candidate explicitly mentions LangGraph and basic AI agent understanding with Python and fluent English, but is very junior with only 8 months of experience, far below the 5+ years required for a senior founding role.
Group ID (Email): alopezgiraldo7@gmail.com
  - Point ID: 203, Score: 0.8472
  - Benchmark Score: 2 | reasoning: Alejandro has strong experience in AI/LLM application development, automation, and Python, with relevant projects involving LangChain and API integrations, but his total relevant experience is slightly under the 5+ years required.
Group ID (Email): Benjahermad@gmail.com
  - Point ID: 274, Score: 0.8471
  - Benchmark Score: 1 | reasoning: The candidate demonstra

Doing a more serious evaluation. Calculating Hit rate, Mean Reciprocal Rank (MRR) and Normalized Discounted Cumulative Gain (NDCG)

In [57]:
import math
import pandas as pd

Q = len(job_descriptions)
hr_thr = 2 # Hit rate relevance threshold
mrr2_thr = 2 # MRR_2 relevance threshold
mrr3_thr = 3 # MRR_3 relevance threshold
top_k = 10

metrics = {
    "hit_rate": {},
    "mrr_2": {}, # Mean Reciprocal Rank with a relevance (Benchmark score) of 2
    "mrr_3": {},
    "ndcg_10": {}, # Normalized Discounted Cumulative Gain at K=10
}

for model_name, config in models_to_test.items():

    hit_rate_accum = 0
    mrr2_accum = 0
    mrr3_accum = 0
    ndcg_accum = 0.0

    print("\n" + "="*80)
    print(f" MODEL: {model_name}")
    print("="*80)

    for job in job_descriptions:
        COLLECTION_NAME = f"test_{model_name}"
        model = loaded_models[model_name]
        
        query_vector = model.encode(job["description"]).tolist()
        
        result = client.query_points_groups(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=top_k,
            group_by="email",
            group_size=1
        )

        has_hit = False
        mrr2_rr = 0.0
        mrr3_rr = 0.0
        dcg = 0.0

        for rank, group in enumerate(result.groups, start=1):
            eval_matches = [e for e in ground_truth_matrix[job["id"]]["evaluations"] if e["email"] == group.id]
            if eval_matches:
                benchmark_score = eval_matches[0]["score"]
                print(f"score {benchmark_score} in job {job["id"]}")
                
                # Calculate DCG for current result
                dcg += benchmark_score / math.log2(rank + 1)
                
                if benchmark_score >= hr_thr:
                    has_hit = True
                
                if benchmark_score >= mrr2_thr and mrr2_rr == 0.0:
                    mrr2_rr = 1.0 / rank
                    
                if benchmark_score >= mrr3_thr and mrr3_rr == 0.0:
                    mrr3_rr = 1.0 / rank

        # Ideal DCG (IDCG) for current job from ground truth
        all_eval_scores = sorted(
            [e["score"] for e in ground_truth_matrix[job["id"]]["evaluations"]],
            reverse=True
        )[:top_k]
        idcg = sum(score / math.log2(rank + 1) for rank, score in enumerate(all_eval_scores, start=1))
        
        ndcg_query = (dcg / idcg) if idcg > 0 else 0.0
        ndcg_accum += ndcg_query

        if has_hit:
            hit_rate_accum += 1
            
        mrr2_accum += mrr2_rr
        mrr3_accum += mrr3_rr

    metrics["hit_rate"][model_name] = hit_rate_accum / Q
    metrics["mrr_2"][model_name] = mrr2_accum / Q
    metrics["mrr_3"][model_name] = mrr3_accum / Q
    metrics["ndcg_10"][model_name] = ndcg_accum / Q

df_metrics = pd.DataFrame(metrics)
df_metrics



 MODEL: GIST-all-MiniLM-L6-v2
score 1 in job jd_ai_agents_engineer
score 2 in job jd_ai_agents_engineer
score 1 in job jd_ai_agents_engineer
score 3 in job jd_ai_agents_engineer
score 1 in job jd_ai_agents_engineer
score 1 in job jd_ai_agents_engineer
score 3 in job jd_ai_agents_engineer
score 1 in job jd_ai_agents_engineer
score 0 in job jd_ai_agents_engineer
score 2 in job jd_ai_agents_engineer
score 1 in job jd_btl_marketing_coordinator
score 1 in job jd_btl_marketing_coordinator
score 0 in job jd_btl_marketing_coordinator
score 2 in job jd_btl_marketing_coordinator
score 1 in job jd_btl_marketing_coordinator
score 1 in job jd_btl_marketing_coordinator
score 2 in job jd_btl_marketing_coordinator
score 1 in job jd_btl_marketing_coordinator
score 0 in job jd_btl_marketing_coordinator
score 0 in job jd_btl_marketing_coordinator
score 1 in job jd_senior_sap_data_analyst
score 1 in job jd_senior_sap_data_analyst
score 1 in job jd_senior_sap_data_analyst
score 1 in job jd_senior_sap_data

,hit_rate,mrr_2,mrr_3,ndcg_10
GIST-all-MiniLM-L6-v2,0.8,0.550000,0.272222,0.606711
bge-base-en-v1.5,1.0,0.733333,0.500000,0.688151
harrier-oss-v1-0.6b,1.0,0.833333,0.500000,0.756539
Qwen3-Embedding-4B,1.0,0.688889,0.550000,0.681531


The harrier-oss-v1-0.6b model gives the best results except in mrr_3 in which Qwen3-Embedding-4B is better. On top of that harrier-oss-v1-0.6b is way lighter than Qwen3-Embedding-4B


| Modelo | Idioma | Zero-shot % | # params | Retrieval % | Max. tokens | Dimensions | Link |
| --- | --- | --- | --- | --- | --- | --- | --- |
| All-MiniLM-L6-v2 | Multi | 96 | 23M | 33.3 | 512 | 384 | https://mteb-leaderboard.hf.space/models/avsolatorio/GIST-all-MiniLM-L6-v2 |
| BAAI/bge-base-en-v1.5 | Multi | 99 | 109M | 38.6 | 512 | 768 | https://mteb-leaderboard.hf.space/models/BAAI/bge-base-en-v1.5 |
| microsoft/harrier-oss-v1-0.6b | Multi | 78 | 596M | 70.75 | 32,768 | 1024 | https://mteb-leaderboard.hf.space/models/microsoft/harrier-oss-v1-0.6b |
| Qwen/Qwen3-Embedding-4B | Multi | 99 | 4B | 69.6 | 32,568 | 2560 | https://mteb-leaderboard.hf.space/models/Qwen/Qwen3-Embedding-4B |